In [3]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os 

load_dotenv()

engine = create_engine (
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

customers = pd.read_sql("select * from customers", engine)
customers.head()
orders = pd.read_sql("select * from orders", engine)
orders.head()

,order_id,customer_id,order_date,status
0,1,498,2025-11-01 22:17:15.349991+00:00,completed
1,2,286,2026-07-20 22:17:15.349991+00:00,completed
2,3,686,2026-05-24 22:17:15.349991+00:00,shipped
3,4,749,2026-05-02 22:17:15.349991+00:00,completed
4,5,211,2025-11-11 22:17:15.349991+00:00,completed


#### Задача 1: найти для каждого клиента дату его первого и последнего заказа

In [4]:
# Сначала группировка и считываение дат
result = (
    orders.groupby('customer_id')['order_date']
    .agg(first_order='min', last_order='max')
    .reset_index()
)

# Приклеивание данных из другой таблицы
result = result.merge(customers, on='customer_id', how='left')

# Вывод нужных колонок
final_table = result[['customer_id', 'first_name', 'last_name', 'first_order', 'last_order']].sort_values('customer_id')
final_table


,customer_id,first_name,last_name,first_order,last_order
0,1,Юрий,Пономарева,2026-01-03 22:17:15.367577+00:00,2026-08-02 22:17:15.362589+00:00
1,2,Феофан,Баранова,2026-01-02 22:17:15.357601+00:00,2026-08-19 22:17:15.372849+00:00
2,3,Елизавета,Рыбаков,2026-05-08 22:17:15.373847+00:00,2026-06-29 22:17:15.354976+00:00
3,4,Никон,Тетерина,2025-10-18 22:17:15.374374+00:00,2026-04-02 22:17:15.372849+00:00
4,5,Валерий,Лукина,2025-09-10 22:17:15.361591+00:00,2026-06-05 22:17:15.355803+00:00
...,...,...,...,...,...
794,796,Никодим,Дроздов,2025-10-27 22:17:15.367577+00:00,2026-05-09 22:17:15.374374+00:00
795,797,Вениамин,Фокина,2025-11-21 22:17:15.369856+00:00,2026-06-08 22:17:15.377171+00:00
796,798,Никифор,Волкова,2026-02-24 22:17:15.359598+00:00,2026-08-10 22:17:15.361591+00:00
797,799,Жанна,Аксенов,2025-10-08 22:17:15.370855+00:00,2026-04-03 22:17:15.355803+00:00


#### Задача 2: для каждого товара найти его ранг по цене внутри своей категории

In [5]:
products = pd.read_sql("select * from products", engine)

products['price_rank'] = (
    products.groupby('category')['price']
    .rank(method='min', ascending=False)
)

final_table = (
    products[
        ['product_id', 'product_name', 'category', 'price', 'price_rank']
    ]
    .sort_values(['category', 'price_rank', 'product_id'])
    .reset_index(drop=True)
)

final_table

,product_id,product_name,category,price,price_rank
0,48,Взаимовыгодная и веб-ориентированная иерархия,Дом и сад,602.22,1.0
1,19,Бизнес-ориентированная и специализированная ми...,Дом и сад,566.31,2.0
2,34,Новая и исполнительная возможность,Дом и сад,532.85,3.0
3,159,Адаптивная и национальная эмуляция,Дом и сад,488.51,4.0
4,122,Ориентированная и объектно-ориентированная инф...,Дом и сад,483.87,5.0
...,...,...,...,...,...
195,32,Органичный и основной эталон,Электроника,70.20,30.0
196,95,Безопасный и оптимальный системный движок,Электроника,66.70,31.0
197,77,Интуитивное и статическое хранилище данных,Электроника,53.29,32.0
198,115,Синхронизированная и региональная ценовая стру...,Электроника,28.28,33.0
